# Spatial Anonymization – Usage Examples


This notebook demonstrates how to use the **spatial anonymization** library to protect survey GPS data while preserving analytical utility.


## Spatial Anonymization: Overview

### What Is Spatial Anonymization?

Spatial anonymization introduces controlled uncertainty into GPS coordinates to protect respondent privacy while preserving useful geographic patterns.

This library implements three common approaches:

- **DHS-style displacement**  
  Random displacement within a fixed maximum distance (optionally urban/rural-specific)

- **Donut geomasking**  
  Enforces a minimum displacement distance to reduce re-identification risk

- **Population-adaptive displacement**  
  Adjusts the displacement radius to ensure a minimum surrounding population

---

## Anonymization Logic

The anonymization process separates privacy policy, geometry, and validation:

1. **Radius Selection (Privacy Envelope)**  
   Defines the uncertainty range (fixed, donut, or population-based)

2. **Point Displacement (Geometry)**  
   Generates a candidate point using a random direction and distance

3. **Constraint Evaluation (Validity)**  
   Ensures the point satisfies spatial rules (e.g., boundaries, water exclusion)

Population defines the anonymization envelope, while constraints determine validity—ensuring predictable runtime, avoiding infinite retries, and enabling modular testing.

---

##  Dataset Description

This notebook uses household survey data from **Malawi** for demonstration purposes, including GPS coordinates and associated metadata.

- **Source:** Malawi Household Survey  
- **Coverage:** Multiple districts  
- **Unit:** Households within enumeration areas (EAs)

### Key Data Fields

- **Identifiers & geography:** `HHID`, `ea_id`, `ea_code`, `district`  
- **Survey metadata:** `interviewDate`, `reside`, `camp`, `hh_a02a`, `hh_a02b`  
- **GPS (Garmin – primary):** `latitude_garmin`, `longitude_garmin`  
- **GPS (Tablet – supplementary):** coordinates, accuracy, altitude, timestamp, weather, satellites  

### Data Characteristics
- Quality metrics support reliability checks  
- Urban/rural classification enables differentiated policies  

---

## Notebook Structure

1. [Python Environment Setup](#setup)  
2. [Working Directory and Data Layout](#working-directory)  
3. [Simple API: Basic Usage](#simple-basic)  
4. [Simple API: Automatic Data Downloads](#simple-download)  
5. [Simple API: Applying Constraints](#simple-constraints)  
6. [Config-Based API: Full Control](#config-based)  
7. [Config-Based API: Loading from YAML](#config-yaml)  
8. [Advanced Usage: User-Provided Spatial Data](#advanced-user-data)

---

## 1. Python Setup <a id='setup'></a>

1. Ensure that you have the package installed by following instructions provided on the GitHub project documentation.
2. Import required libraries 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

from spatial_anonymization.api import (
    anonymize_survey_coordinates, 
    validate_input_schema
    )
from spatial_anonymization.diagnostics import diagnostics, visualization

from spatial_anonymization.config.params import DataSourcesConfig

# Import configuration presets from config_presets module
from spatial_anonymization.config.presets import DHS, print_preset

## 2. Working Directory and Inputs Setup

Define global path variables for your working directory and data folders. These paths are used to:
- load input datasets (e.g. survey coordinates, boundaries)  
- store downloaded resources (e.g. population rasters)  
- save outputs and diagnostics  

Centralizing paths improves clarity, portability, and reproducibility.  
Use a consistent naming convention: directories (`DIR_*`) and files (`FILE_*`).

### Example

```python
from pathlib import Path

# Base directory
DIR_WORK = Path("/path/to/your/project")

# Data directories
DIR_DATA = DIR_WORK / "data"
FILE_MWI_COORDS = DIR_WORK / "data/mwi/ihs_coords.dta"


In [8]:
# ======================================
# SETUP INPUT PATHS CONSTANTS
# ======================================
# Base working directory
DIR_WORK = Path("/Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/")

# Input data directories
DIR_DATA = DIR_WORK / "data"

# ========================
# INPUT FILES FOR MALAWI
# ========================
# MW working directory
DIR_MWI = DIR_DATA / "mwi"
DIR_MWI_GEOSPATIAL = DIR_MWI / "geospatial"

# Survey coordinates file
FILE_MWI_IHS6 = DIR_MWI / "test-coords.csv"


# MWI adm3 shapefile
FILE_MWI_ADM3 = DIR_MWI_GEOSPATIAL / "adm4-wgs84.shp"

# Pre-downloaded population raster file
FILE_MWI_POPULATION_RASTER = DIR_MWI_GEOSPATIAL / "mwi_worldpop_2025.tif"

# Pre-downloaded water bodies shapefile (from OSM)
FILE_MWI_WATER_BODIES = DIR_MWI_GEOSPATIAL / "MWI_water_bodies_osm.shp"

## 3. Input Data Preparation

Before running anonymization, users must prepare both coordinate data and any required geospatial inputs.

### 3.1 Coordinates

Ensure that the input dataset is cleaned and correctly structured:

- Longitude and latitude in decimal degrees (WGS84)  
- Numeric values with no missing entries  

Specify:
- longitude and latitude column names  
- urban/rural classification (if available)  
- labels for urban and rural areas  

---

### 3.2 Administrative Boundaries

Administrative boundaries are used to constrain displacement.

Users can:
- provide their own boundary data (e.g. shapefile or GeoJSON), or  
- allow the package to download boundaries automatically  

If downloading, users must specify:
- **country ISO code** (e.g. `MWI`)  
- **administrative level** (e.g. adm0–adm3)  

By default, the package attempts to download from:
- **[geoBoundaries](https://www.geoboundaries.org)**
- fallback to **[GADM](https://gadm.org/data.html)**

---

### 3.3 Population Data (Optional)

Population data is required only for population-based anonymization methods.

Users can:
- provide their own population raster, or  
- allow the package to download one automatically  

By default, the package uses:

- **[WorldPop (100m resolution, R2025A v1)](https://hub.worldpop.org/geodata/listing?id=135)**
  
---

In [4]:
# ============================================================
# SPECIFY INPUT DATA SCHEMA
# ============================================================
lat = "latitude_tablet"
lon = "longitude_tablet"
urban_rural = "urb_rur"
urban_value = "URBAN"
rural_value = "RURAL"

In [5]:
# ======================================
# VALIDATE INPUT DATA
# =======================================
# Load the data
df_mwi_ihs = pd.read_csv(FILE_MWI_IHS6)

# Check the data
validate_input_schema(df_mwi_ihs, lat_col=lat, lon_col=lon,
                      urban_rural_col=urban_rural, urban_value="URBAN",
                      rural_value="RURAL")


All Input Data Schema Validations Passed!


In [12]:
# ======================================
# Geospatial Input Layers 
# =======================================
# Admin boundaries file defined in Working Directory and Inputs Setup
print(f"Admin boundaries file: {FILE_MWI_ADM3}")

# Population raster file defined in Working Directory and Inputs Setup
print(f"Population raster file: {FILE_MWI_POPULATION_RASTER}")

Admin boundaries file: /Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/data/mwi/geospatial/adm4-wgs84.shp
Population raster file: /Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/data/mwi/geospatial/mwi_worldpop_2025.tif


## 4. Configuring Spatial Anonymization

Before running anonymization, users need to make a few key decisions about how privacy should be enforced. These choices determine the displacement method, distance rules, and any spatial constraints.

The package supports two approaches:

### Option-1: Preset Configurations (Recommended)

Use predefined configurations for common methods such as DHS-style displacement.  
These presets reflect standard practices and require minimal setup, with optional overrides.

### Option-2. Custom Configuration

Define parameters manually for full control over:
- displacement method and distance thresholds  
- urban/rural rules  
- population thresholds  
- spatial constraints  

### Key Decisions

At a minimum, decide:
- which anonymization method to use  
- whether to differentiate urban and rural areas  
- whether to apply population thresholds  
- which constraints to enforce (e.g., administrative boundaries)

> **In this notebook, we begin with a vanilla DHS configuration**, which includes:
> - urban displacement up to 2 km  
> - rural displacement up to 5 km, with ~10% of points up to 10 km  
> - enforcement of administrative boundaries as the only spatial constraint  
>
> For administrative constraints, users must choose the appropriate boundary level.  
> In Malawi, administrative levels range from **adm0 (country)** to **adm3 (traditional authority)**, with enumeration areas being more granular.  
> In this example, displacement is constrained within **adm3 boundaries**.

In [ ]:
# ======================================
# SPATIAL ANONYMIZATION CONFIGS 
# =======================================
# We will use the DHS preset configuration for this example, 
# which is suitable for survey data with urban/rural classification. The DHS preset includes parameters such as maximum displacement distances for urban and rural areas, and the number of random points to generate for each cluster.
METHOD = "dhs"

# Review the parameters for the DHS preset
print("DHS Preset Parameters:")
print_preset(DHS)

## 5. Running the Survey Anonymization

Call `anonymize_survey_coordinates()` to displace the GPS coordinates. At a minimum, you must provide:

### Required Inputs

| Input | Description |
|---|---|
| `df` | A pandas DataFrame containing the survey coordinates |
| `country_iso3` | ISO 3166-1 alpha-3 country code (e.g. `"MWI"`) |
| `lon_col` / `lat_col` | Names of the longitude and latitude columns in your DataFrame |
| `output_dir` | Directory where output files (CSV and GeoJSON) will be saved |

### Recommended Inputs

| Input | Description |
|---|---|
| `urban_rural_col` | Column indicating urban/rural classification |
| `urban_value` / `rural_value` | Labels used in that column (e.g. `"URBAN"`, `"RURAL"`) |
| `admin_boundaries` | Path to an administrative boundary shapefile for constraining displacement |
| `method` | Anonymization method — `"dhs"` (default), `"donut"`, or `"population_adaptive"` |

### Optional Inputs

| Input | Description |
|---|---|
| `water_boundaries` | Water body shapefile to avoid displacing points into water |
| `population_raster` | Population raster for population-adaptive methods or post-displacement extraction |
| `n_workers` | Number of parallel processes (`1` for reproducibility) |
| `max_attempts` | Retry limit per point (default: 100) |
| `extract_displaced_population` | Whether to query population at displaced locations |

If `admin_boundaries` is not provided, the package downloads boundaries automatically using `country_iso3`.

In [ ]:
# ======================================
# ANONYMIZE THE DATA
# =======================================
df_anonymized = anonymize_survey_coordinates(
    df=df_mwi_ihs,
    lon_col=lon,
    lat_col=lat,
    urban_rural_col="urb_rur",
    urban_value=urban_value,
    rural_value=rural_value,
    method="dhs",
    country_iso3="MWI",
    output_dir=DIR_MWI / "ihs6-2024-25",
    admin_boundaries=FILE_MWI_ADM4,
    water_boundaries=FILE_MWI_WATER_BODIES,
    population_raster=FILE_MWI_POPULATION_RASTER,
    n_workers=12,
    admin_level=4,
    max_attempts=100,
    extract_displaced_population=True
)

╔═════════════════════════════════════════════════════════════════════════╗
║                       ANONYMIZATION CONFIGURATION                       ║
╠═════════════════════════════════════════════════════════════════════════╣
║  Preset: dhs  (+6 overrides)                                            ║
╠═════════════════════════════════════════════════════════════════════════╣
║  DISPLACEMENT METHOD                                                    ║
║   method                   dhs          Random displacement within max radius║
╠═════════════════════════════════════════════════════════════════════════╣
║  DISTANCE THRESHOLDS                                                    ║
║   urban_max_distance       2000 m       Max displacement for urban points║
║   rural_max_distance       5000 m       Max displacement for rural points║
║   rural_extra_max          10000 m      Extended rural displacement     ║
║   rural_extra_prop         10.0%        Fraction of rural points extended║
║   

In [8]:
df_anonymized.head()

,hh_id,urb_rur,ea_code,latitude_tablet,longitude_tablet,lon_anon,lat_anon,distance_km,attempts,max_radius_assigned,pop_within_1000m
0,41ed290e6cdf4e61acf4ba371c49b49a,RURAL,10101002,-9.713875,33.280730,33.318744,-9.734621,4.760810,1,5000,517
1,4ab2906c97734193b5dd7635dfa5896a,RURAL,10101002,-9.712385,33.343880,33.335980,-9.723054,1.464281,1,5000,513
2,56851caf37314f5b90beb74ffcbb0fa4,RURAL,10101002,-9.713875,33.280730,33.270123,-9.728631,2.004599,1,5000,521
3,577bcae775754b4d8fa2aa474e612046,RURAL,10101002,-9.638638,33.256250,33.256710,-9.618433,2.235434,1,5000,450
4,6c540f4f3521446b9084deec5f600bc8,RURAL,10101002,-9.637543,33.249335,33.228198,-9.662663,3.619598,1,5000,284


In [9]:
df_eth = pd.read_csv("/Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/data/tests/eth/coords.csv")

In [10]:
df_eth.query('ea_id == "41402010100102"')

,ea_id,urban_rural,ea_ctr_lat,ea_ctr_lon,lat_dd_mod,lon_dd_mod


In [12]:
adm = "/Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/data/tests/eth/shp_adm2/eth_admbnda_adm2_csa_bofed_20190827.shp"
outdir = Path("/Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/data/tests/eth/outputs")
df_eth_anon = anonymize_survey_coordinates(
    df=df_eth,
    lon_col='ea_ctr_lon',
    lat_col='ea_ctr_lat',
    urban_rural_col="urban_rural",
    urban_value="URBAN",
    rural_value="RURAL",
    method="dhs",
    country_iso3="ETH",
    output_dir=outdir,
    admin_boundaries=adm,
    n_workers=12,
    max_attempts=100,
    extract_displaced_population=False
)

╔═════════════════════════════════════════════════════════════════════════╗
║                       ANONYMIZATION CONFIGURATION                       ║
╠═════════════════════════════════════════════════════════════════════════╣
║  Preset: dhs  (+5 overrides)                                            ║
╠═════════════════════════════════════════════════════════════════════════╣
║  DISPLACEMENT METHOD                                                    ║
║   method                   dhs          Random displacement within max radius║
╠═════════════════════════════════════════════════════════════════════════╣
║  DISTANCE THRESHOLDS                                                    ║
║   urban_max_distance       2000 m       Max displacement for urban points║
║   rural_max_distance       5000 m       Max displacement for rural points║
║   rural_extra_max          10000 m      Extended rural displacement     ║
║   rural_extra_prop         10.0%        Fraction of rural points extended║
║   

In [13]:
df_eth_anon.head()

,ea_id,urban_rural,ea_ctr_lat,ea_ctr_lon,lat_dd_mod,lon_dd_mod,lon_anon,lat_anon,distance_km,attempts,max_radius_assigned
0,10101088800910,RURAL,14.255856,37.816607,14.258308,37.809656,37.808378,14.215542,4.547949,1,5000
1,10102088801010,RURAL,14.380982,38.157330,14.375913,38.154890,38.126756,14.369430,3.536630,1,5000
2,10103088800709,RURAL,14.111058,38.421738,14.110021,38.425236,38.443815,14.123797,2.769357,1,5000
3,10104010100101,URBAN,14.200314,38.340043,14.196226,38.355781,38.356063,14.205144,1.809883,1,2000
4,10104088800405,RURAL,14.108654,38.267188,14.109313,38.265418,38.260653,14.105465,0.788956,1,5000


In [17]:
displacement_stats = diagnostics.displacement_statistics(
    df=df_eth_anon,
    lon_col="ea_ctr_lon",
    lat_col="ea_ctr_lat",
    anon_lat_col="lat_anon",
    anon_lon_col="lon_anon",
    urban_col="urban_rural",
)


 Distance Stats: Original vs. Displaced Coordinates

--- all ---
  Stat    Distance (Km)
  ------  -------------
  count        535.0000
  mean           2.6205
  median         1.8664
  min            0.1976
  max            9.6302
  p90            4.7938
  p95            5.0099
  p99            9.1033

--- RURAL ---
  Stat                 Distance (Km)
  -------------------  -------------
  count                     297.0000
  mean                        3.6530
  median                      3.6325
  min                         0.2003
  max                         9.6302
  p90                         4.9779
  p95                         8.0131
  p99                         9.3770
  check_max<=10km             1.0000
  share_>5km                  0.0909
  share_>10km                 0.0000
  expected_share_>5km         0.0100

--- URBAN ---
  Stat            Distance (Km)
  --------------  -------------
  count                238.0000
  mean                   1.3321
  median          

In [18]:
displacement_stats = diagnostics.displacement_statistics(
    df=df_eth_anon,
    lon_col="ea_ctr_lon",
    lat_col="ea_ctr_lat",
    anon_lat_col="lat_dd_mod",
    anon_lon_col="lon_dd_mod",
    urban_col="urban_rural",
)


 Distance Stats: Original vs. Displaced Coordinates

--- all ---
  Stat    Distance (Km)
  ------  -------------
  count        535.0000
  mean           1.9225
  median         1.4992
  min            0.0011
  max            8.6570
  p90            4.2887
  p95            4.7103
  p99            7.6836

--- RURAL ---
  Stat                 Distance (Km)
  -------------------  -------------
  count                     297.0000
  mean                        2.6793
  median                      2.5296
  min                         0.0051
  max                         8.6570
  p90                         4.6799
  p95                         5.4378
  p99                         8.1631
  check_max<=10km             1.0000
  share_>5km                  0.0572
  share_>10km                 0.0000
  expected_share_>5km         0.0100

--- URBAN ---
  Stat            Distance (Km)
  --------------  -------------
  count                238.0000
  mean                   0.9781
  median          

In [ ]:
# ===================================================================
# DIAGNOSTICS: CHECK ANONYMIZED POINTS FALL WITHIN ADMIN BOUNDARIES
# ===================================================================
gdf_adm4 = gpd.read_file(FILE_MWI_ADM4)
admin_check = diagnostics.admin_boundary_check(df_anonymized, lon_col=lon, lat_col=lat, admin_gdf=gdf_adm4)

In [ ]:
# ============================================
# DIAGNOSTICS: DISPLACEMENT STATISTICS
# ============================================
displacement_stats = diagnostics.displacement_statistics(
    df=df_anonymized,
    lon_col=lon,
    lat_col=lat,
    urban_col="urb_rur",
)
